# Worm Moebius

In this notebook, we implement the worm algorithm for the Moebius code for $d = 2 p$ with $p$ odd prime. 

In [1]:
from jax.typing import ArrayLike
import jax.numpy as jnp
from functools import partial
import numpy as np
from typing import Dict
from coherentinfo.moebius_two_odd_prime import MoebiusCodeTwoOddPrime
from coherentinfo.errormodel import ErrorModelLindbladTwoOddPrime
from coherentinfo.cohinfo_sf_method import (
    run_worm_moebius_sf,
)
import time
import json
import os
import jax
n_cpus = os.cpu_count()
n_used_cpus = n_cpus
print("Number of CPUs available: {}".format(n_cpus))
print("Number of used CPUs: {}".format(n_used_cpus))
jax.config.update('jax_num_cpu_devices', n_used_cpus)
# jax.config.update("jax_log_compiles", True)
# Devices assumed by JAX
print(jax.devices())

Number of CPUs available: 16
Number of used CPUs: 16
[CpuDevice(id=0), CpuDevice(id=1), CpuDevice(id=2), CpuDevice(id=3), CpuDevice(id=4), CpuDevice(id=5), CpuDevice(id=6), CpuDevice(id=7), CpuDevice(id=8), CpuDevice(id=9), CpuDevice(id=10), CpuDevice(id=11), CpuDevice(id=12), CpuDevice(id=13), CpuDevice(id=14), CpuDevice(id=15)]


In [ ]:
moebius_setup = {"length": 3, "width": 3, "p": 3}
num_samples = 50
master_error_seed = 14
master_worm_seed = 20
worm_setup = {}
worm_setup["num_worms"] = 10 * n_used_cpus
worm_setup["burn_in_steps"] = moebius_setup["length"] * 15000  # 1000  # 2000
worm_setup["max_worm_steps"] = worm_setup["burn_in_steps"] + \
    moebius_setup["length"] * 10000  # 5000
keys_setup = {"master_error_seed": master_error_seed,
              "master_worm_seed": master_worm_seed}
gamma_t = 0.2

In [3]:
moebius_code = MoebiusCodeTwoOddPrime(
    length=moebius_setup["length"], width=moebius_setup["width"], d=2 * moebius_setup["p"])
em_lindblad = ErrorModelLindbladTwoOddPrime(
    moebius_code.num_edges, d=2 * moebius_setup["p"], gamma_t=gamma_t
)
print(f"Single error probabilities: {em_lindblad.get_probabilities()}")

Single error probabilities: [0.69740236 0.1367651  0.01363117 0.00180542 0.01363096 0.13676497]


In [4]:
new_worm_state, unique_syndromes, syndrome_counts = run_worm_moebius_sf(
    gamma_t=gamma_t,
    syndrome_id="plaquette",
    moebius_setup=moebius_setup,
    num_samples=num_samples,
    worm_setup=worm_setup,
    keys_setup=keys_setup
)

In [ ]:
new_worm_state